In [0]:
# Célula 1 — setup
import uuid
from datetime import datetime, timezone
from pyspark.sql import functions as F, Window

TABELA_BRONZE_PRODUTOS = "workspace.squad1.bronze_rafael_bruna_ecommerce_produtos"
TABELA_BRONZE_CATEGORIAS = "workspace.squad1.bronze_rafael_bruna_ecommerce_categorias"
TABELA_SILVER_PRODUTOS = "workspace.squad1.silver_rafael_bruna_ecommerce_produtos"
TABELA_SILVER_CATEGORIAS = "workspace.squad1.silver_rafael_bruna_ecommerce_categorias"
TABELA_LOGS = "workspace.squad1.dq_monitoring_logs"  # NOME PROVISÓRIO — pendente resposta da squad

df_produtos_bronze = spark.table(TABELA_BRONZE_PRODUTOS)
df_categorias_bronze = spark.table(TABELA_BRONZE_CATEGORIAS)

print("produtos no Bronze:", df_produtos_bronze.count(), "| categorias no Bronze:", df_categorias_bronze.count())

In [0]:
# Célula 2 — regras de ecommerce_produtos, uma coluna booleana por linha
def aplicar_regras_produtos(df, df_categorias_ref):
    df = df.withColumn("_dup_sku", F.count("*").over(Window.partitionBy("sku", "bronze_source_file")))
    categorias_validas = df_categorias_ref.select(F.col("id_categoria").alias("_cat")).distinct()
    subcategorias = (df_categorias_ref.filter(F.col("id_categoria_pai").isNotNull())
                      .select(F.col("id_categoria").alias("_subcat")).distinct())

    df = (df
        .withColumn("flag_sku_nao_nulo", F.col("sku").isNotNull())
        .withColumn("flag_sku_unico", F.col("_dup_sku") == 1)
        .withColumn("flag_nome_produto_preenchido",
                    F.col("nome_produto").isNotNull() & (F.trim(F.col("nome_produto")) != ""))
        .withColumn("flag_preco_positivo", F.col("preco_lista") > 0)
        .withColumn("flag_preco_dentro_limite", F.col("preco_lista") <= 5000)
        .withColumn("flag_unidade_valida", F.col("unidade_medida").isin(["un", "kg", "g", "L", "ml"]))
        .withColumn("flag_marca_preenchida",
                    F.col("nome_marca").isNotNull() & (F.trim(F.col("nome_marca")) != ""))
        .withColumn("flag_is_ativo_preenchido", F.col("is_ativo").isNotNull())
        .withColumn("flag_venda_90d", F.lit(None).cast("boolean"))
        .drop("_dup_sku"))

    df = (df.join(categorias_validas, df["id_categoria"] == F.col("_cat"), "left")
            .withColumn("flag_categoria_existe", F.col("_cat").isNotNull()).drop("_cat"))
    df = (df.join(subcategorias, df["id_categoria"] == F.col("_subcat"), "left")
            .withColumn("flag_pertence_subcategoria", F.col("_subcat").isNotNull()).drop("_subcat"))

    return df.withColumn("silver_processed_at", F.current_timestamp())


df_produtos_silver = aplicar_regras_produtos(df_produtos_bronze, df_categorias_bronze)

In [0]:
# Célula 3 — regras de ecommerce_categorias, uma coluna booleana por linha
def aplicar_regras_categorias(df, df_produtos_ref):
    w_id = Window.partitionBy("id_categoria", "bronze_source_file")
    w_grp = Window.partitionBy(F.coalesce(F.col("id_categoria_pai"), F.lit(-1)), "nome_categoria", "bronze_source_file")

    pais_validos = df.select(F.col("id_categoria").alias("_pai")).distinct()
    produtos_por_cat = df_produtos_ref.select(F.col("id_categoria").alias("_tem_produto")).distinct()
    raizes_com_sub = (df.filter(F.col("id_categoria_pai").isNotNull())
                       .select(F.col("id_categoria_pai").alias("_raiz_sub")).distinct())
    avos = (df.filter(F.col("id_categoria_pai").isNotNull())
             .select(F.col("id_categoria").alias("_id_pai"), F.col("id_categoria_pai").alias("_avo")).distinct())

    df = (df
        .withColumn("_dup_id", F.count("*").over(w_id))
        .withColumn("_dup_nome", F.count("*").over(w_grp))
        .withColumn("flag_id_nao_nulo", F.col("id_categoria").isNotNull())
        .withColumn("flag_id_unico", F.col("_dup_id") == 1)
        .withColumn("flag_nome_preenchido",
                    F.col("nome_categoria").isNotNull() & (F.trim(F.col("nome_categoria")) != ""))
        .withColumn("flag_sem_autociclo",
                    F.when(F.col("id_categoria_pai").isNull(), F.lit(True))
                     .otherwise(F.col("id_categoria") != F.col("id_categoria_pai")))
        .withColumn("flag_nome_unico_no_grupo", F.col("_dup_nome") == 1)
        .withColumn("flag_nome_valido", ~F.col("nome_categoria").rlike(r"[<>&]|(?i)script"))
        .drop("_dup_id", "_dup_nome"))

    df = (df.join(pais_validos, df["id_categoria_pai"] == F.col("_pai"), "left")
            .withColumn("flag_pai_existe", F.col("id_categoria_pai").isNull() | F.col("_pai").isNotNull())
            .drop("_pai"))
    df = (df.join(produtos_por_cat, df["id_categoria"] == F.col("_tem_produto"), "left")
            .withColumn("flag_subcategoria_tem_produto",
                        F.when(F.col("id_categoria_pai").isNotNull(), F.col("_tem_produto").isNotNull()))
            .drop("_tem_produto"))
    df = (df.join(raizes_com_sub, df["id_categoria"] == F.col("_raiz_sub"), "left")
            .withColumn("flag_raiz_tem_subcategoria",
                        F.when(F.col("id_categoria_pai").isNull(), F.col("_raiz_sub").isNotNull()))
            .drop("_raiz_sub"))
    df = (df.join(avos, df["id_categoria_pai"] == F.col("_id_pai"), "left")
            .withColumn("flag_profundidade_ok", F.col("_avo").isNull())
            .drop("_id_pai", "_avo"))

    return df.withColumn("silver_processed_at", F.current_timestamp())


df_categorias_silver = aplicar_regras_categorias(df_categorias_bronze, df_produtos_bronze)

In [0]:
# Célula 4 — grava a Silver inteira (overwrite direto, sem contagem intermediária)
df_produtos_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABELA_SILVER_PRODUTOS)
df_categorias_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TABELA_SILVER_CATEGORIAS)

# única fonte de verdade: reler a tabela depois de gravar
print("Silver produtos:", spark.table(TABELA_SILVER_PRODUTOS).count())
print("Silver categorias:", spark.table(TABELA_SILVER_CATEGORIAS).count())

In [0]:
# Célula 5 — log de qualidade (ainda sem gravar — dq_monitoring_logs pendente de decisão da squad)
SCHEMA_LOGS = ("run_id string, tabela string, regra string, status string, severidade string, "
               "qtd_registros_falhos long, qtd_registros_total long, timestamp_execucao timestamp, arquivo_origem string")

def gerar_log(df_flagado, tabela_nome, mapa_regras, run_id, agora):
    linhas = []
    for regra, coluna, severidade in mapa_regras:
        aplicaveis = df_flagado.filter(F.col(coluna).isNotNull())
        total = aplicaveis.count()
        falhas = aplicaveis.filter(~F.col(coluna)).count()
        linhas.append((run_id, tabela_nome, regra, "FAIL" if falhas else "PASS",
                        severidade, falhas, total, agora, "real-time-data (batches do dia)"))
    return linhas

MAPA_PRODUTOS = [
    ("sku_nao_nulo", "flag_sku_nao_nulo", "Critica"),
    ("sku_unico", "flag_sku_unico", "Critica"),
    ("nome_produto_preenchido", "flag_nome_produto_preenchido", "Critica"),
    ("categoria_existe", "flag_categoria_existe", "Critica"),
    ("unidade_valida", "flag_unidade_valida", "Aviso"),
    ("is_ativo_preenchido", "flag_is_ativo_preenchido", "Critica"),
    ("preco_positivo", "flag_preco_positivo", "Critica"),
    ("preco_dentro_limite", "flag_preco_dentro_limite", "Aviso"),
    ("marca_preenchida", "flag_marca_preenchida", "Aviso"),
    ("pertence_subcategoria", "flag_pertence_subcategoria", "Aviso"),
]
MAPA_CATEGORIAS = [
    ("id_nao_nulo", "flag_id_nao_nulo", "Critica"),
    ("id_unico", "flag_id_unico", "Critica"),
    ("nome_preenchido", "flag_nome_preenchido", "Critica"),
    ("pai_existe", "flag_pai_existe", "Critica"),
    ("sem_autociclo", "flag_sem_autociclo", "Critica"),
    ("nome_unico_no_grupo", "flag_nome_unico_no_grupo", "Aviso"),
    ("nome_valido", "flag_nome_valido", "Aviso"),
    ("subcategoria_tem_produto", "flag_subcategoria_tem_produto", "Aviso"),
    ("raiz_tem_subcategoria", "flag_raiz_tem_subcategoria", "Aviso"),
    ("profundidade_ok", "flag_profundidade_ok", "Critica"),
]

run_id = str(uuid.uuid4())
agora = datetime.now(timezone.utc)
linhas_log = (gerar_log(df_produtos_silver, "ecommerce_produtos", MAPA_PRODUTOS, run_id, agora)
              + gerar_log(df_categorias_silver, "ecommerce_categorias", MAPA_CATEGORIAS, run_id, agora))

df_logs = spark.createDataFrame(linhas_log, schema=SCHEMA_LOGS)
df_logs.orderBy("status", "tabela").show(30, truncate=False)

# df_logs.write.format("delta").mode("append").saveAsTable(TABELA_LOGS)  # comentado — pendente decisão da squad